 ## Using LLMs for inference in few-shot prompting

 This code was written mainly be me, however Claude was used to trouble shoot, specifically for the API timeout error that requires asyncio and the LengthFinishReasonError. Please refer to the LLM_labeling.ipynb notebook for specific comments on how the code functions as they share the same overall approach.
 
 The results displayed in this notebook are not the final outputs used in the thesis, they are older and use the default SEQEVAL metrics, which is not correct.

In [ ]:
from dotenv import load_dotenv
import os
import asyncio
import json
import os
import time
from openai import AsyncAzureOpenAI
from openai import LengthFinishReasonError
from openai import AzureOpenAI
from pydantic import BaseModel
from seqeval.metrics import classification_report
from pathlib import Path


In [2]:
load_dotenv(".env")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
api_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")

In [3]:
def get_list_of_sentences(file_path):
    """this returns a list of lists for sentences and list of lists for labels"""
    current_sent = []
    current_label = []
    all_sents = []
    all_labels = []

    with open(file_path, "r", encoding="utf-8") as infile:
        lines = infile.readlines()
        for line in lines:
            line = line.strip()
            if line == "":
                if len(current_sent) > 0:
                    all_sents.append(current_sent)
                    all_labels.append(current_label)
                    current_sent = []
                    current_label = []
            else:
                splitted = line.split("\t")
                token_part = splitted[0]
                label_part = splitted[1]
                current_sent.append(token_part)
                current_label.append(label_part)

        if len(current_sent) > 0:
            all_sents.append(current_sent)
            all_labels.append(current_label)
    
    return all_sents, all_labels


def results_classified(predicted_file, gold_file= r"C:\Users\M.Walavalkar\OneDrive - IBFD\Desktop\thesis-ner-manya-explore\datasets\my_data\final_dataset_18th_March.conll"):
    # gold_file = r"C:\Users\M.Walavalkar\OneDrive - IBFD\Desktop\thesis-ner-manya-explore\datasets\my_data\final_dataset_18th_March.conll"
    gold_sents, gold_labels = get_list_of_sentences(gold_file)
    llm_sents, llm_labels= get_list_of_sentences(predicted_file)
    y_true, y_pred = gold_labels, llm_labels
    report = classification_report(y_true, y_pred, digits=2)
    print(report)
    return
    
    # print(len(llm_sent_2), len(llm_labels_2))
    

In [ ]:
def ner_to_conll(token_map, entities):
    """this converts a token map + entity list to CoNLL BIO format."""
    entity_list = []
    new_list = []
    for thingies in entities:
        sent_dict = {}
        sentence_dict = {}
        sentence_dict["sentence_index"] = thingies.sent_index
        sentence_dict["entity_info"] = { "token": thingies.token_text,
                                        "token_index": thingies.token_index,
                                        "predicted_label": str(thingies.label).split(".")[1],
                                        "entity_index": thingies.entity_index}
        
        entity_list.append(sentence_dict)

    result = []
    prev_ent_index = None
    for dictionaries in token_map:
        prev_ent_index = None
        mapped_tokens = dictionaries["token_map"]
        if result:
             result.append("\n")
        for i in sorted(mapped_tokens.keys()):
            token = mapped_tokens[i]
            matched = False
            for entry in entity_list:
                    current_sent = dictionaries["sentence_number"]
                    if entry["sentence_index"] == current_sent and entry["entity_info"]["token_index"] == i:
                            label = entry["entity_info"]["predicted_label"]
                            ent_idx = entry["entity_info"]["entity_index"]
                            if ent_idx  == prev_ent_index:
                                prefix = "I"
                            else:
                                prefix = "B"
                                prev_ent_index = ent_idx
                            
                            matched = True
                            result.append((token, f"{prefix}-{label}"))
                            break
            if not matched:
                result.append((token, "O"))
                prev_ent_index = None
    return result

In [8]:
gold_sents, gold_labels = get_list_of_sentences(r"C:\Users\M.Walavalkar\OneDrive - IBFD\Desktop\thesis-ner-manya-explore\datasets\my_data\final_dataset_1st_may.conll")
# llm_sents, llm_labels = get_list_of_sentences(r"gptner_fewshot_1.conll")

### Defining client

In [9]:
def create_client() -> AsyncAzureOpenAI:
    return AsyncAzureOpenAI(
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
        api_version = "2024-12-01-preview"
    )

client = create_client()

model = "gpt-5.4-mini"
deployment ="gpt-5.4-mini"

# CREATING FINAL DATASET VARIABLE

In [ ]:
test_file = Path(r"C:\Users\M.Walavalkar\OneDrive - IBFD\Desktop\thesis-ner-manya-explore\datasets\my_data\final_dataset_1st_may.conll")
tokens = []
labels = []
tokens_sentence = []
labels_sentence = []
token_count = []
with open(test_file, "r") as f:
    for line in f.readlines():
        if len(line) > 1:
            line = line.strip()
            token, label = line.split("\t")
            tokens_sentence.append(token)
            labels_sentence.append(label)
        else:
            if tokens_sentence:
                tokens.append(tokens_sentence)
                labels.append(labels_sentence)
                
            tokens_sentence = []
            labels_sentence = []
    if tokens_sentence:
        tokens.append(tokens_sentence)
        labels.append(labels_sentence)

print(len(tokens))

final_sents = []
counter = 0
for tok in tokens:
    new_sentence = " ".join(tok)
    final_sents.append([new_sentence])

with open("sentences.json", "w") as json_file:
    json.dump(final_sents, json_file)

656


# GPT-NER ##TAGGING@@ <TECHNIQUE>

In [ ]:
def ent_dict_to_conll(entity_dictionary, sentence_list, output_file):
    """
    this writes the entity dictionary to conll with IOB2 labeling
    """
    with open(output_file, "w", encoding="utf-8") as outfile:
        for sent_index, sentence in enumerate(sentence_list):
            tokens = sentence.split(" ")
            
            sentence_entities = [
                e for e in entity_dictionary 
                if e.get("sentence_index") == sent_index
            ]

            for i, tok in enumerate(tokens):
                matched = False
                for entry in sentence_entities:
                    if entry["ent_start"] <= i <= entry["ent_end"]:
                        prefix = "B" if i == entry["ent_start"] else "I"
                        outfile.write(f"{tok}\t{prefix}-{entry['label']}\n")
                        matched = True
                        break
                if not matched:
                    outfile.write(f"{tok}\tO\n")

            outfile.write("\n")


In [ ]:
class Entity(BaseModel):
    sentence_index: int     ## sentence number
    full_sentence_with_markings: str        # exact surface form, e.g. "New York"

class NERResponse(BaseModel):
    entities: list[Entity]

def load_sentences(path):
    raw = json.load(open(path))
    return [group[0] for group in raw]

In [ ]:
## few shot + DETAILED descriptions and all examples
SYSTEM_PROMPT = """You are a Legal and Tax NER tagger. Given a sentence, return it exactly as provided but with every entity marked using the GPT-NER inline format:

    ##entity text@@ <LABEL>

where ## opens the entity span and @@ closes it, followed by a space, the label in angle brackets, and a space before the next token. Tokens that are not part of any entity are left unmarked. Never leave an entity span unclosed.

─── LABELS ───

PERSON
  Covers: named or specifically referenced individuals in their human capacity, including judges, advocates general, named parties, and definite singular references resolvable to one specific individual in context (e.g., Mr. Bosal, Advocate General Konll, the applicant). Spans include personal names with attached titles or roles. 
	Excludes: generic plurals (taxpayers, judges), corporate or legal persons even when named like individuals (use ORG), and unresolved generic role references (a taxpayer in this situation).

ORG
  Covers: named non-judicial organizations — companies, agencies, supranational bodies, tax authorities, professional bodies (e.g., Bosal Holding BV, European Commission, OECD, HMRC). Spans include the full proper name with name-constitutive modifiers (European in European Commission, Internal in Internal Revenue Service) and standard legal-form suffixes (BV, plc, GmbH). 
	Excludes: courts and tribunals (use COURT), jurisdictional modifiers that merely localize a generic institution, generic descriptions without naming (the company, the holding), and entity-type concepts (holding company, subsidiary) (use TAX_CONCEPT).

GPE
  Covers: specific, named geopolitical entities — countries, cities, regions, supranational unions with concrete identity (e.g., Belgium, Brussels, European Union, EU, California). Spans cover the bare name; articles are included only when part of the official name (The Hague). 
	Excludes: demonyms and adjectival forms (Belgian, Dutch, European) entirely are never tagged on their own and are stripped when they modify another entity, abstract role-based references (use JURISDICTION), and geographic regions without political identity (Europe as a continent).

LAW
  Covers: named statutory or treaty-level instruments referred to as a whole, without article or section pinpointing (e.g., TFEU, EC Treaty, Parent-Subsidiary Directive, Income Tax Act 2007). Spans include the full official title, standard acronyms, definite back-references that unambiguously resolve to a previously named instrument (the Treaty, the Directive, the Code), and years embedded in a statutory title. 
  Excludes: pinpoint references (use PROVISION), case law and judgments, and generic legal-concept terms (the law, legislation).

DATE
  Covers: absolute or relative temporal expressions denoting a point or period (e.g., 18 September 2003, 1990, 2019). Spans cover only the date expression itself; contextualizing nouns that label what kind of period it is are stripped. 
	Excludes: durations not anchored to a calendar point (for five years), vague temporal references (recently, previously, last year), and dates embedded inside another entity's name (the 2007 in Income Tax Act 2007 stays inside the LAW span).

TAX_TYPE
  Covers: named tax instruments as they appear in context (e.g., corporate income tax, withholding tax, VAT).
	Exclude: jurisdictional modifiers, generic nouns alone (tax, rate), and verbal or metaphorical uses (withholding documents, a tax on patience).

TAX_CONCEPT
  Covers: domain terms that aren't named instruments: legal doctrines, planning mechanisms, behaviors, and economic concepts (e.g., transfer pricing, arm's length, permanent establishment).
	Excludes: accounting-standard vocabulary such as GAAP, IFRS, or depreciation.

PROVISION
  Covers: specific articles, sections, paragraphs, or sub-paragraphs within a named instrument, including the host instrument when cited together (e.g., Article 49 TFEU, Article 4(1) of the Parent-Subsidiary Directive, Section 3(2), paragraph 3). Spans cover the full citation as a single entity — pinpoint plus host instrument plus subdivisions — and coordinated pinpoints sharing one host as one span (Articles 43 and 49 EC). 
	Excludes: the host instrument cited alone without a pinpoint (use LAW), case-paragraph references (paragraph 23 of the judgment), and recital references unless project-scoped in.

JURISDICTION
  Covers: abstract or role-based references to a state in its legal or fiscal capacity, where the specific country is unnamed or generalized (e.g., Member State, residence state, source state, host state, contracting state, third country). Spans include the role descriptor with role-defining qualifiers (the Member State concerned, the State of residence). 
	Excludes: named countries even when filling such a role (use GPE) and purely geographic terms without legal-role meaning.

COURT
  Covers: judicial bodies, tribunals, and adjudicative offices (e.g., Court of Justice, European Court of Justice, CJEU, Hoge Raad, Bundesfinanzhof). Spans include the full proper name with name-constitutive modifiers (European in European Court of Justice), standard acronyms, and case-specific unnamed references (the referring court, the national court). 
	Excludes: jurisdictional modifiers that merely localize a generic court, individual judges or advocates general by name (use PERSON), and metaphorical uses (the court of public opinion).
    
─── CORRECT EXAMPLES ───

Example 1:
Text: "The outcome of this case was quite predictable and must be considered
correct"
Expected output: "The outcome of this case was quite predictable and must be
considered correct"

Example 2:
Text: " Residents are subject to world-wide taxation by virtue of Article 2 of
the LIR."
Expected output: "Residents are subject to world-wide taxation by virtue of
##Article 2 of the LIR@@ <PROVISION> .”

Example 3:
Text: “On 15 March 2017, the European Commission issued a decision requiring Luxembourg to recover the unlawful state aid.”
Expected output: “On ##15 March 2017@@<DATE>, the ##European Commission@@<ORG> issued a decision requiring ##Luxembourg@@<GPE> to recover the unlawful state aid.”

Example 4:
Text: “The Dutch corporate income tax does not apply to profits attributed to a permanent establishment in cases involving transfer pricing adjustments.”
Expected output: “The Dutch ##corporate income tax@@<TAX_TYPE> does not apply to profits attributed to a ##permanent establishment@@<TAX_CONCEPT> in cases involving ##transfer pricing@@<TAX_CONCEPT> adjustments.”

Example 5:
Text: “Advocate General Kokott argued before the Court of Justice that the residence state must grant relief from double taxation.”
Expected output: “##Advocate General Kokott@@<PERSON> argued before the ##Court of Justice@@<COURT> that the ##residence state@@<JURISDICTION> must grant relief from ##double taxation@@<TAX_CONCEPT>.”

Example 6:
Text: “Under Article 49 TFEU, Belgium may not impose withholding tax on dividends paid to a parent company in another Member State”
Expected output: “Under ##Article 49 TFEU@@<PROVISION>, ##Belgium@@<GPE> may not impose ##withholding tax@@<TAX_TYPE> on dividends paid to a parent company in another ##Member State@@<JURISDICTION>”

─── BOUNDARY RULES ───
1. Tag the minimal named span. Jurisdictional or descriptive modifiers before a TAX_TYPE are not part of the entity: 'Dutch ## corporate income tax @@ <TAX_TYPE>'.
2. When a provision reference includes the law name, tag the entire reference as a single PROVISION: '## Article 49 TFEU @@ <PROVISION>'.
3. Do not double-tag overlapping spans. Each token belongs to at most one entity.
4. Reproduce the original sentence exactly — do not alter, reorder, or omit any word. Only insert ## and @@ markers with their labels."""


In [ ]:
## few shot -- simple definition + examples

SYSTEM_PROMPT = """You are a Legal and Tax NER tagger. Given a sentence, return it exactly as provided but with every entity marked using the GPT-NER inline format:

    ##entity text@@ <LABEL>

where ## opens the entity span and @@ closes it, followed by a space, the label in angle brackets, and a space before the next token. Tokens that are not part of any entity are left unmarked. Never leave an entity span unclosed.

─── LABELS ───

PERSON
  Covers: people, including judges, advocates, parties, and named individuals.

ORG
  Covers: companies, agencies, institutions, and international bodies.
  Excludes: courts and tribunals (use COURT).

GPE
  Covers: specific named countries, cities, states, and regions.
  Excludes: abstract jurisdictional references (use JURISDICTION).

LAW
  Covers: named laws, treaties, directives, regulations, and conventions.
  Excludes: specific article or section references (use PROVISION).

DATE
  Covers: absolute dates, relative dates, and date ranges or periods.

TAX_TYPE
  Covers: named tax instruments as they appear in context.
  Excludes: jurisdictional modifiers preceding the instrument; generic nouns alone; verbal or metaphorical uses.

TAX_CONCEPT
  Covers: domain-specific terms that are not named tax instruments — legal doctrines, planning mechanisms, economic concepts, and behavioural descriptions.
  Excludes: accounting-standard vocabulary.

PROVISION
  Covers: specific articles, sections, paragraphs, or clauses within a law, including the law name when it appears as part of the reference.

JURISDICTION
  Covers: abstract or role-based jurisdictional references.
  Excludes: specific named countries or cities (use GPE).

COURT
  Covers: courts, tribunals, and judicial bodies, including generic references.

    
─── CORRECT EXAMPLES ───

Example 1:
Text: "The outcome of this case was quite predictable and must be considered
correct"
Expected output: "The outcome of this case was quite predictable and must be
considered correct"

Example 2:
Text: " Residents are subject to world-wide taxation by virtue of Article 2 of
the LIR."
Expected output: "Residents are subject to world-wide taxation by virtue of
##Article 2 of the LIR@@ <PROVISION> .”

Example 3:
Text: “On 15 March 2017, the European Commission issued a decision requiring Luxembourg to recover the unlawful state aid.”
Expected output: “On ##15 March 2017@@<DATE>, the ##European Commission@@<ORG> issued a decision requiring ##Luxembourg@@<GPE> to recover the unlawful state aid.”

Example 4:
Text: “The Dutch corporate income tax does not apply to profits attributed to a permanent establishment in cases involving transfer pricing adjustments.”
Expected output: “The Dutch ##corporate income tax@@<TAX_TYPE> does not apply to profits attributed to a ##permanent establishment@@<TAX_CONCEPT> in cases involving ##transfer pricing@@<TAX_CONCEPT> adjustments.”

Example 5:
Text: “Advocate General Kokott argued before the Court of Justice that the residence state must grant relief from double taxation.”
Expected output: “##Advocate General Kokott@@<PERSON> argued before the ##Court of Justice@@<COURT> that the ##residence state@@<JURISDICTION> must grant relief from ##double taxation@@<TAX_CONCEPT>.”

Example 6:
Text: “Under Article 49 TFEU, Belgium may not impose withholding tax on dividends paid to a parent company in another Member State”
Expected output: “Under ##Article 49 TFEU@@<PROVISION>, ##Belgium@@<GPE> may not impose ##withholding tax@@<TAX_TYPE> on dividends paid to a parent company in another ##Member State@@<JURISDICTION>”

─── BOUNDARY RULES ───
1. Tag the minimal named span. Jurisdictional or descriptive modifiers before a TAX_TYPE are not part of the entity: 'Dutch ## corporate income tax @@ <TAX_TYPE>'.
2. When a provision reference includes the law name, tag the entire reference as a single PROVISION: '## Article 49 TFEU @@ <PROVISION>'.
3. Do not double-tag overlapping spans. Each token belongs to at most one entity.
4. Reproduce the original sentence exactly — do not alter, reorder, or omit any word. Only insert ## and @@ markers with their labels."""


In [ ]:
def get_ent_spans_from_string(marked_sentence, sentence_index):
    tokens = marked_sentence.split(" ")
    parsed = []
    inside = False
    real_index = 0
    current_span_tokens = []
    span_start = None

    for token in tokens:
        if token.startswith("##") and token.endswith("@@"):
            span_start = real_index
            current_span_tokens = [token[2:-2]]
            real_index += 1
        elif token.startswith("##"):
            span_start = real_index
            current_span_tokens = [token[2:]]
            inside = True
            real_index += 1
        elif inside and token.endswith("@@"):
            current_span_tokens.append(token[:-2])
            inside = False
            real_index += 1
        elif inside:
            current_span_tokens.append(token)
            real_index += 1
        elif token.startswith("<") and token.endswith(">") and current_span_tokens:
            parsed.append({
                "sentence_index": sentence_index,
                "ent_start": span_start,
                "ent_end": real_index - 1,
                "ent_text": " ".join(current_span_tokens),
                "label": token[1:-1],
            })
            current_span_tokens = []
            span_start = None
        else:
            real_index += 1

    if inside:
        print(f"Unclosed entity in sentence!!! {sentence_index}: {marked_sentence}")

    return parsed

async def run_ner(client: AsyncAzureOpenAI, sentence: str, sentence_index: int) -> tuple[dict, dict]:
    try:
        response = await client.beta.chat.completions.parse(
            model="gpt-5.4-mini",
            max_completion_tokens=4096,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": sentence},
            ],
            response_format=NERResponse,
        )
        usage = {
            "prompt_tokens": response.usage.prompt_tokens,
            "completion_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens,
        }
        raw = {
            "sentence_index": sentence_index,
            "marked_sentences": [e.full_sentence_with_markings for e in response.choices[0].message.parsed.entities]
        }
        return raw, usage
    except LengthFinishReasonError as e:
        raise e

async def main():
    sentences = load_sentences("sentences.json")

    tasks = [run_ner(client, sentence, i) for i, sentence in enumerate(sentences)]
    results = await asyncio.gather(*tasks)

    raw_results = []
    total_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
    for raw, usage in results:
        raw_results.append(raw)
        for key in total_usage:
            total_usage[key] += usage[key]

    with open("raw_marked_output.json", "w", encoding="utf-8") as f:
        json.dump(raw_results, f, indent=2, ensure_ascii=False)

    all_parsed_entities = []
    for raw in raw_results:
        sentence_index = raw["sentence_index"]
        for marked_sentence in raw["marked_sentences"]:
            parsed = get_ent_spans_from_string(marked_sentence, sentence_index)
            all_parsed_entities.extend(parsed)

    print(f"Found {len(all_parsed_entities)} entities")
    print(f"  Prompt tokens:     {total_usage['prompt_tokens']:,}")
    print(f"  Completion tokens: {total_usage['completion_tokens']:,}")
    print(f"  Total tokens:      {total_usage['total_tokens']:,}")

    ent_dict_to_conll(all_parsed_entities, sentences, "gptner_fewshot_1_may_with_simple_def.conll")
    return all_parsed_entities

In [14]:
gptner_results = await main()
print(len(gptner_results))

Found 1605 entities
  Prompt tokens:     633,353
  Completion tokens: 54,302
  Total tokens:      687,655
1605


### Results

In [15]:
results_classified("gptner_fewshot_1_may_with_simple_def.conll", gold_file =  r"C:\Users\M.Walavalkar\OneDrive - IBFD\Desktop\thesis-ner-manya-explore\datasets\my_data\final_dataset_1st_may.conll")

              precision    recall  f1-score   support

       COURT       0.34      0.73      0.46       106
        DATE       0.70      0.58      0.63       132
         GPE       0.68      0.65      0.67       299
JURISDICTION       0.45      0.49      0.47       153
         LAW       0.21      0.46      0.29        65
         ORG       0.48      0.36      0.41       221
      PERSON       0.79      0.82      0.80       186
   PROVISION       0.37      0.64      0.47        87
 TAX_CONCEPT       0.38      0.11      0.18       353
    TAX_TYPE       0.54      0.36      0.43        86

   micro avg       0.51      0.48      0.49      1688
   macro avg       0.50      0.52      0.48      1688
weighted avg       0.52      0.48      0.48      1688



In [61]:
results_classified("gptner_fewshot_1_may.conll", gold_file =  r"C:\Users\M.Walavalkar\OneDrive - IBFD\Desktop\thesis-ner-manya-explore\datasets\my_data\final_dataset_1st_may.conll")

              precision    recall  f1-score   support

       COURT       0.41      0.68      0.51       106
        DATE       0.65      0.54      0.59       132
         GPE       0.70      0.64      0.67       299
JURISDICTION       0.59      0.72      0.65       153
         LAW       0.18      0.42      0.25        65
         ORG       0.42      0.36      0.39       221
      PERSON       0.61      0.55      0.58       186
   PROVISION       0.40      0.57      0.47        87
 TAX_CONCEPT       0.33      0.07      0.12       353
    TAX_TYPE       0.51      0.29      0.37        86

   micro avg       0.50      0.45      0.47      1688
   macro avg       0.48      0.48      0.46      1688
weighted avg       0.50      0.45      0.45      1688



# Index:Token Dictionary Marking

In [16]:
from enum import Enum
class Label(str, Enum):
    PERSON = 1
    ORG = 2
    GPE = 3
    LAW = 4
    DATE = 5
    TAX_TYPE = 6
    TAX_CONCEPT = 7
    PROVISION = 8
    JURISDICTION = 9
    COURT = 10

class Entity(BaseModel):
    token_text: str
    token_index: int   # key from the input token map
    label:  Label  # PER, LOC, or ORG #make e num -- to bound the values the variable can take
    entity_index: int      # the entity index so that it can be converted into B/I spans
    sent_index: int #sentence id

class NERResponse(BaseModel):
    entities: list[Entity]


In [17]:
#simple descriptions + exmaples
SYSTEM_PROMPT = """You are a Legal and Tax NER classifier. Given a JSON object containing a sentence_number and a token_map (mapping token indices to tokens), identify all named entities and return a JSON array of entity tokens.

─── LABELS ───

PERSON
  Covers: people, including judges, advocates, parties, and named individuals.

ORG
  Covers: companies, agencies, institutions, and international bodies.
  Excludes: courts and tribunals (use COURT).

GPE
  Covers: specific named countries, cities, states, and regions.
  Excludes: abstract jurisdictional references (use JURISDICTION).

LAW
  Covers: named laws, treaties, directives, regulations, and conventions.
  Excludes: specific article or section references (use PROVISION).

DATE
  Covers: absolute dates, relative dates, and date ranges or periods.

TAX_TYPE
  Covers: named tax instruments as they appear in context.
  Excludes: jurisdictional modifiers preceding the instrument; generic nouns alone; verbal or metaphorical uses.

TAX_CONCEPT
  Covers: domain-specific terms that are not named tax instruments — legal doctrines, planning mechanisms, economic concepts, and behavioural descriptions.
  Excludes: accounting-standard vocabulary.

PROVISION
  Covers: specific articles, sections, paragraphs, or clauses within a law, including the law name when it appears as part of the reference.

JURISDICTION
  Covers: abstract or role-based jurisdictional references.
  Excludes: specific named countries or cities (use GPE).

COURT
  Covers: courts, tribunals, and judicial bodies, including generic references.


─── EXAMPLES ───

Example 1 (no entities):
Input: {"sentence_number": 0, "token_map": {"0": "The", "1": "outcome", "2": "of", "3": "this", "4": "case", "5": "was", "6": "quite", "7": "predictable", "8": "."}}
Output: []

Example 2 (PROVISION):
Input: {"sentence_number": 1, "token_map": {"0": "Residents", "1": "are", "2": "subject", "3": "to", "4": "world-wide", "5": "taxation", "6": "by", "7": "virtue", "8": "of", "9": "Article", "10": "2", "11": "of", "12": "the", "13": "LIR", "14": "."}}
Output: [
  {"sent_index": 1, "token_index": 9, "token_text": "Article", "entity_index": 0, "label": 8},
  {"sent_index": 1, "token_index": 10, "token_text": "2", "entity_index": 0, "label": 8},
  {"sent_index": 1, "token_index": 11, "token_text": "of", "entity_index": 0, "label": 8},
  {"sent_index": 1, "token_index": 12, "token_text": "the", "entity_index": 0, "label": 8},
  {"sent_index": 1, "token_index": 13, "token_text": "LIR", "entity_index": 0, "label": 8}
]

Example 3 (ORG + GPE + DATE):
Input: {"sentence_number": 2, "token_map": {"0": "On", "1": "15", "2": "March", "3": "2017", "4": ",", "5": "the", "6": "European", "7": "Commission", "8": "issued", "9": "a", "10": "decision", "11": "requiring", "12": "Luxembourg", "13": "to", "14": "recover", "15": "the", "16": "unlawful", "17": "state", "18": "aid", "19": "."}}
Output: [
  {"sent_index": 2, "token_index": 1, "token_text": "15", "entity_index": 0, "label": 5},
  {"sent_index": 2, "token_index": 2, "token_text": "March", "entity_index": 0, "label": 5},
  {"sent_index": 2, "token_index": 3, "token_text": "2017", "entity_index": 0, "label": 5},
  {"sent_index": 2, "token_index": 6, "token_text": "European", "entity_index": 1, "label": 2},
  {"sent_index": 2, "token_index": 7, "token_text": "Commission", "entity_index": 1, "label": 2},
  {"sent_index": 2, "token_index": 12, "token_text": "Luxembourg", "entity_index": 2, "label": 3}
]

Example 4 (TAX_TYPE + TAX_CONCEPT + jurisdictional modifier stripped):
Input: {"sentence_number": 3, "token_map": {"0": "The", "1": "Dutch", "2": "corporate", "3": "income", "4": "tax", "5": "does", "6": "not", "7": "apply", "8": "to", "9": "profits", "10": "attributed", "11": "to", "12": "a", "13": "permanent", "14": "establishment", "15": "in", "16": "cases", "17": "involving", "18": "transfer", "19": "pricing", "20": "adjustments", "21": "."}}
Output: [
  {"sent_index": 3, "token_index": 2, "token_text": "corporate", "entity_index": 0, "label": 6},
  {"sent_index": 3, "token_index": 3, "token_text": "income", "entity_index": 0, "label": 6},
  {"sent_index": 3, "token_index": 4, "token_text": "tax", "entity_index": 0, "label": 6},
  {"sent_index": 3, "token_index": 13, "token_text": "permanent", "entity_index": 1, "label": 7},
  {"sent_index": 3, "token_index": 14, "token_text": "establishment", "entity_index": 1, "label": 7},
  {"sent_index": 3, "token_index": 18, "token_text": "transfer", "entity_index": 2, "label": 7},
  {"sent_index": 3, "token_index": 19, "token_text": "pricing", "entity_index": 2, "label": 7
]

Example 5 (COURT + PERSON + JURISDICTION + TAX_CONCEPT):
Input: {"sentence_number": 4, "token_map": {"0": "Advocate", "1": "General", "2": "Kokott", "3": "argued", "4": "before", "5": "the", "6": "Court", "7": "of", "8": "Justice", "9": "that", "10": "the", "11": "residence", "12": "state", "13": "must", "14": "grant", "15": "relief", "16": "from", "17": "double", "18": "taxation", "19": "."}}
Output: [
  {"sent_index": 4, "token_index": 0, "token_text": "Advocate", "entity_index": 0, "label": 1},
  {"sent_index": 4, "token_index": 1, "token_text": "General", "entity_index": 0, "label": 1},
  {"sent_index": 4, "token_index": 2, "token_text": "Kokott", "entity_index": 0, "label": 1},
  {"sent_index": 4, "token_index": 6, "token_text": "Court", "entity_index": 1, "label": 10},
  {"sent_index": 4, "token_index": 7, "token_text": "of", "entity_index": 1, "label": 10},
  {"sent_index": 4, "token_index": 8, "token_text": "Justice", "entity_index": 1, "label": 10},
  {"sent_index": 4, "token_index": 11, "token_text": "residence", "entity_index": 2, "label": 9},
  {"sent_index": 4, "token_index": 12, "token_text": "state", "entity_index": 2, "label": 9},
  {"sent_index": 4, "token_index": 17, "token_text": "double", "entity_index": 3, "label": 7},
  {"sent_index": 4, "token_index": 18, "token_text": "taxation", "entity_index": 3, "label": 7}
]

Example 6 (PROVISION + GPE + TAX_TYPE + JURISDICTION):
Input: {"sentence_number": 5, "token_map": {"0": "Under", "1": "Article", "2": "49", "3": "TFEU", "4": ",", "5": "Belgium", "6": "may", "7": "not", "8": "impose", "9": "withholding", "10": "tax", "11": "on", "12": "dividends", "13": "paid", "14": "to", "15": "a", "16": "parent", "17": "company", "18": "in", "19": "another", "20": "Member", "21": "State", "22": "."}}
Output: [
  {"sent_index": 5, "token_index": 1, "token_text": "Article", "entity_index": 0, "label": 8},
  {"sent_index": 5, "token_index": 2, "token_text": "49", "entity_index": 0, "label": 8},
  {"sent_index": 5, "token_index": 3, "token_text": "TFEU", "entity_index": 0, "label": 8},
  {"sent_index": 5, "token_index": 5, "token_text": "Belgium", "entity_index": 1, "label": 3},
  {"sent_index": 5, "token_index": 9, "token_text": "withholding", "entity_index": 2, "label": 6},
  {"sent_index": 5, "token_index": 10, "token_text": "tax", "entity_index": 2, "label": 6},
  {"sent_index": 5, "token_index": 20, "token_text": "Member", "entity_index": 3, "label": 9},
  {"sent_index": 5, "token_index": 21, "token_text": "State", "entity_index": 3, "label": 9}

  ─── BOUNDARY RULES ───

1. Tag the minimal named span. Jurisdictional modifiers before a TAX_TYPE are not part of the entity: in 'Dutch corporate income tax', only 'corporate income tax' is TAX_TYPE.
2. When a provision reference includes the law name, tag the entire reference as a single PROVISION.
3. Each token belongs to at most one entity.
4. Only return tokens that are part of entities. Do not return O labels.

─── OUTPUT FORMAT ───

Return ONLY a JSON array. Each element represents one entity token:
{
  "sent_index": <sentence_number from input>,
  "token_index": <token position from token_map>,
  "token_text": <exact token string>,
  "entity_index": <integer grouping multi-token entities, starting at 0>,
  "label": "<one of the following integers representing the respective label: PERSON = 1, ORG = 2, GPE = 3, LAW = 4, DATE = 5, TAX_TYPE = 6, TAX_CONCEPT = 7, PROVISION = 8, JURISDICTION = 9, COURT = 10>"
}

Consecutive tokens sharing the same entity_index form one entity span. Increment entity_index for each new entity.
"""  

In [ ]:
#detailed + intext examples
SYSTEM_PROMPT = """You are a Legal and Tax NER classifier. Given a JSON object containing a sentence_number and a token_map (mapping token indices to tokens), identify all named entities and return a JSON array of entity tokens.

─── LABELS ───

PERSON
  Covers: named or specifically referenced individuals in their human capacity, including judges, advocates general, named parties, and definite singular references resolvable to one specific individual in context (e.g., Mr. Bosal, Advocate General Konll, the applicant). Spans include personal names with attached titles or roles. 
	Excludes: generic plurals (taxpayers, judges), corporate or legal persons even when named like individuals (use ORG), and unresolved generic role references (a taxpayer in this situation).

ORG
  Covers: named non-judicial organizations — companies, agencies, supranational bodies, tax authorities, professional bodies (e.g., Bosal Holding BV, European Commission, OECD, HMRC). Spans include the full proper name with name-constitutive modifiers (European in European Commission, Internal in Internal Revenue Service) and standard legal-form suffixes (BV, plc, GmbH). 
	Excludes: courts and tribunals (use COURT), jurisdictional modifiers that merely localize a generic institution, generic descriptions without naming (the company, the holding), and entity-type concepts (holding company, subsidiary) (use TAX_CONCEPT).

GPE
  Covers: specific, named geopolitical entities — countries, cities, regions, supranational unions with concrete identity (e.g., Belgium, Brussels, European Union, EU, California). Spans cover the bare name; articles are included only when part of the official name (The Hague). 
	Excludes: demonyms and adjectival forms (Belgian, Dutch, European) entirely are never tagged on their own and are stripped when they modify another entity, abstract role-based references (use JURISDICTION), and geographic regions without political identity (Europe as a continent).

LAW
  Covers: named statutory or treaty-level instruments referred to as a whole, without article or section pinpointing (e.g., TFEU, EC Treaty, Parent-Subsidiary Directive, Income Tax Act 2007). Spans include the full official title, standard acronyms, definite back-references that unambiguously resolve to a previously named instrument (the Treaty, the Directive, the Code), and years embedded in a statutory title. 
  Excludes: pinpoint references (use PROVISION), case law and judgments, and generic legal-concept terms (the law, legislation).

DATE
  Covers: absolute or relative temporal expressions denoting a point or period (e.g., 18 September 2003, 1990, 2019). Spans cover only the date expression itself; contextualizing nouns that label what kind of period it is are stripped. 
	Excludes: durations not anchored to a calendar point (for five years), vague temporal references (recently, previously, last year), and dates embedded inside another entity's name (the 2007 in Income Tax Act 2007 stays inside the LAW span).

TAX_TYPE
  Covers: named tax instruments as they appear in context (e.g., corporate income tax, withholding tax, VAT).
	Exclude: jurisdictional modifiers, generic nouns alone (tax, rate), and verbal or metaphorical uses (withholding documents, a tax on patience).

TAX_CONCEPT
  Covers: domain terms that aren't named instruments: legal doctrines, planning mechanisms, behaviors, and economic concepts (e.g., transfer pricing, arm's length, permanent establishment).
	Excludes: accounting-standard vocabulary such as GAAP, IFRS, or depreciation.

PROVISION
  Covers: specific articles, sections, paragraphs, or sub-paragraphs within a named instrument, including the host instrument when cited together (e.g., Article 49 TFEU, Article 4(1) of the Parent-Subsidiary Directive, Section 3(2), paragraph 3). Spans cover the full citation as a single entity — pinpoint plus host instrument plus subdivisions — and coordinated pinpoints sharing one host as one span (Articles 43 and 49 EC). 
	Excludes: the host instrument cited alone without a pinpoint (use LAW), case-paragraph references (paragraph 23 of the judgment), and recital references unless project-scoped in.

JURISDICTION
  Covers: abstract or role-based references to a state in its legal or fiscal capacity, where the specific country is unnamed or generalized (e.g., Member State, residence state, source state, host state, contracting state, third country). Spans include the role descriptor with role-defining qualifiers (the Member State concerned, the State of residence). 
	Excludes: named countries even when filling such a role (use GPE) and purely geographic terms without legal-role meaning.

COURT
  Covers: judicial bodies, tribunals, and adjudicative offices (e.g., Court of Justice, European Court of Justice, CJEU, Hoge Raad, Bundesfinanzhof). Spans include the full proper name with name-constitutive modifiers (European in European Court of Justice), standard acronyms, and case-specific unnamed references (the referring court, the national court). 
	Excludes: jurisdictional modifiers that merely localize a generic court, individual judges or advocates general by name (use PERSON), and metaphorical uses (the court of public opinion).

─── EXAMPLES ───

Example 1 (no entities):
Input: {"sentence_number": 0, "token_map": {"0": "The", "1": "outcome", "2": "of", "3": "this", "4": "case", "5": "was", "6": "quite", "7": "predictable", "8": "."}}
Output: []

Example 2 (PROVISION):
Input: {"sentence_number": 1, "token_map": {"0": "Residents", "1": "are", "2": "subject", "3": "to", "4": "world-wide", "5": "taxation", "6": "by", "7": "virtue", "8": "of", "9": "Article", "10": "2", "11": "of", "12": "the", "13": "LIR", "14": "."}}
Output: [
  {"sent_index": 1, "token_index": 9, "token_text": "Article", "entity_index": 0, "label": 8},
  {"sent_index": 1, "token_index": 10, "token_text": "2", "entity_index": 0, "label": 8},
  {"sent_index": 1, "token_index": 11, "token_text": "of", "entity_index": 0, "label": 8},
  {"sent_index": 1, "token_index": 12, "token_text": "the", "entity_index": 0, "label": 8},
  {"sent_index": 1, "token_index": 13, "token_text": "LIR", "entity_index": 0, "label": 8}
]

Example 3 (ORG + GPE + DATE):
Input: {"sentence_number": 2, "token_map": {"0": "On", "1": "15", "2": "March", "3": "2017", "4": ",", "5": "the", "6": "European", "7": "Commission", "8": "issued", "9": "a", "10": "decision", "11": "requiring", "12": "Luxembourg", "13": "to", "14": "recover", "15": "the", "16": "unlawful", "17": "state", "18": "aid", "19": "."}}
Output: [
  {"sent_index": 2, "token_index": 1, "token_text": "15", "entity_index": 0, "label": 5},
  {"sent_index": 2, "token_index": 2, "token_text": "March", "entity_index": 0, "label": 5},
  {"sent_index": 2, "token_index": 3, "token_text": "2017", "entity_index": 0, "label": 5},
  {"sent_index": 2, "token_index": 6, "token_text": "European", "entity_index": 1, "label": 2},
  {"sent_index": 2, "token_index": 7, "token_text": "Commission", "entity_index": 1, "label": 2},
  {"sent_index": 2, "token_index": 12, "token_text": "Luxembourg", "entity_index": 2, "label": 3}
]

Example 4 (TAX_TYPE + TAX_CONCEPT + jurisdictional modifier stripped):
Input: {"sentence_number": 3, "token_map": {"0": "The", "1": "Dutch", "2": "corporate", "3": "income", "4": "tax", "5": "does", "6": "not", "7": "apply", "8": "to", "9": "profits", "10": "attributed", "11": "to", "12": "a", "13": "permanent", "14": "establishment", "15": "in", "16": "cases", "17": "involving", "18": "transfer", "19": "pricing", "20": "adjustments", "21": "."}}
Output: [
  {"sent_index": 3, "token_index": 2, "token_text": "corporate", "entity_index": 0, "label": 6},
  {"sent_index": 3, "token_index": 3, "token_text": "income", "entity_index": 0, "label": 6},
  {"sent_index": 3, "token_index": 4, "token_text": "tax", "entity_index": 0, "label": 6},
  {"sent_index": 3, "token_index": 13, "token_text": "permanent", "entity_index": 1, "label": 7},
  {"sent_index": 3, "token_index": 14, "token_text": "establishment", "entity_index": 1, "label": 7},
  {"sent_index": 3, "token_index": 18, "token_text": "transfer", "entity_index": 2, "label": 7},
  {"sent_index": 3, "token_index": 19, "token_text": "pricing", "entity_index": 2, "label": 7}
]

Example 5 (COURT + PERSON + JURISDICTION + TAX_CONCEPT):
Input: {"sentence_number": 4, "token_map": {"0": "Advocate", "1": "General", "2": "Kokott", "3": "argued", "4": "before", "5": "the", "6": "Court", "7": "of", "8": "Justice", "9": "that", "10": "the", "11": "residence", "12": "state", "13": "must", "14": "grant", "15": "relief", "16": "from", "17": "double", "18": "taxation", "19": "."}}
Output: [
  {"sent_index": 4, "token_index": 0, "token_text": "Advocate", "entity_index": 0, "label": 1},
  {"sent_index": 4, "token_index": 1, "token_text": "General", "entity_index": 0, "label": 1},
  {"sent_index": 4, "token_index": 2, "token_text": "Kokott", "entity_index": 0, "label": 1},
  {"sent_index": 4, "token_index": 6, "token_text": "Court", "entity_index": 1, "label": 10},
  {"sent_index": 4, "token_index": 7, "token_text": "of", "entity_index": 1, "label": 10},
  {"sent_index": 4, "token_index": 8, "token_text": "Justice", "entity_index": 1, "label": 10},
  {"sent_index": 4, "token_index": 11, "token_text": "residence", "entity_index": 2, "label": 9},
  {"sent_index": 4, "token_index": 12, "token_text": "state", "entity_index": 2, "label": 9},
  {"sent_index": 4, "token_index": 17, "token_text": "double", "entity_index": 3, "label": 7},
  {"sent_index": 4, "token_index": 18, "token_text": "taxation", "entity_index": 3, "label": 7}
]

Example 6 (PROVISION + GPE + TAX_TYPE + JURISDICTION):
Input: {"sentence_number": 5, "token_map": {"0": "Under", "1": "Article", "2": "49", "3": "TFEU", "4": ",", "5": "Belgium", "6": "may", "7": "not", "8": "impose", "9": "withholding", "10": "tax", "11": "on", "12": "dividends", "13": "paid", "14": "to", "15": "a", "16": "parent", "17": "company", "18": "in", "19": "another", "20": "Member", "21": "State", "22": "."}}
Output: [
  {"sent_index": 5, "token_index": 1, "token_text": "Article", "entity_index": 0, "label": 8},
  {"sent_index": 5, "token_index": 2, "token_text": "49", "entity_index": 0, "label": 8},
  {"sent_index": 5, "token_index": 3, "token_text": "TFEU", "entity_index": 0, "label": 8},
  {"sent_index": 5, "token_index": 5, "token_text": "Belgium", "entity_index": 1, "label": 3},
  {"sent_index": 5, "token_index": 9, "token_text": "withholding", "entity_index": 2, "label": 6},
  {"sent_index": 5, "token_index": 10, "token_text": "tax", "entity_index": 2, "label": 6},
  {"sent_index": 5, "token_index": 20, "token_text": "Member", "entity_index": 3, "label": 9},
  {"sent_index": 5, "token_index": 21, "token_text": "State", "entity_index": 3, "label": 9}
]

  ─── BOUNDARY RULES ───

1. Tag the minimal named span. Jurisdictional modifiers before a TAX_TYPE are not part of the entity: in 'Dutch corporate income tax', only 'corporate income tax' is TAX_TYPE.
2. When a provision reference includes the law name, tag the entire reference as a single PROVISION.
3. Each token belongs to at most one entity.
4. Only return tokens that are part of entities. Do not return O labels.

─── OUTPUT FORMAT ───

Return ONLY a JSON array. Each element represents one entity token:
{
  "sent_index": <sentence_number from input>,
  "token_index": <token position from token_map>,
  "token_text": <exact token string>,
  "entity_index": <integer grouping multi-token entities, starting at 0>,
  "label": "<one of the following integers representing the respective label: PERSON = 1, ORG = 2, GPE = 3, LAW = 4, DATE = 5, TAX_TYPE = 6, TAX_CONCEPT = 7, PROVISION = 8, JURISDICTION = 9, COURT = 10>"
}

Consecutive tokens sharing the same entity_index form one entity span. Increment entity_index for each new entity.
"""

In [ ]:
def load_sentences(path):
    raw = json.load(open(path))
    return [group[0] for group in raw]

def build_token_map_lists(sentences):
    """this converts each sentence string into a numbered token map dict."""
    list_of_mappings = []
    for i, sentence in enumerate(sentences):
        tokens = sentence.split(" ")
        list_of_mappings.append({
            "sentence_number": i,
            "token_map": {j: token for j, token in enumerate(tokens)},
        })
    return list_of_mappings

async def run_ner(client, token_map):
    response = await client.beta.chat.completions.parse(
        model=deployment,
        temperature=0.3,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": json.dumps(token_map)},
        ],
        response_format=NERResponse,
    )
    usage = {
        "prompt_tokens": response.usage.prompt_tokens,
        "completion_tokens": response.usage.completion_tokens,
        "total_tokens": response.usage.total_tokens,
    }
    return response.choices[0].message.parsed, usage


async def main():
    sentences = load_sentences("sentences.json")
    token_maps = build_token_map_lists(sentences)

    # client = create_client()
    tasks = [run_ner(client, tm) for tm in token_maps]
    results = await asyncio.gather(*tasks)

    all_entities = []
    total_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
    for ner_response, usage in results:
        all_entities.extend(ner_response.entities)
        for key in total_usage:
            total_usage[key] += usage[key]

    print(f"Found {len(all_entities)} entities:")
    print(all_entities)
    for entity in all_entities:
        print(f"  [{entity.sent_index}] {entity.token_text} ({entity.label})")

    print(f"\nToken usage across {len(results)} requests:")
    print(f"  Prompt tokens:     {total_usage['prompt_tokens']:,}")
    print(f"  Completion tokens: {total_usage['completion_tokens']:,}")
    print(f"  Total tokens:      {total_usage['total_tokens']:,}")

    return all_entities

In [ ]:
tokind_dictionary_fewshot = await main()
print(len(tokind_dictionary_fewshot))

In [20]:
## get all token_maps
sentences = load_sentences("sentences.json")
token_maps = build_token_map_lists(sentences)

In [21]:
results = ner_to_conll(token_maps, tokind_dictionary_fewshot)
with open("dictionary_fewshot_may_with_simple_def.conll", "w", encoding ="utf-8") as outfile:
    for element in results:
        if len(element) == 2:
             t, l = element
             outfile.write(f"{t}\t{l}\n")
        else:
             outfile.write(element)

In [22]:
results_classified("dictionary_fewshot_may_with_simple_def.conll", gold_file =  r"C:\Users\M.Walavalkar\OneDrive - IBFD\Desktop\thesis-ner-manya-explore\datasets\my_data\final_dataset_1st_may.conll")

              precision    recall  f1-score   support

       COURT       0.27      0.67      0.39       106
        DATE       0.73      0.70      0.72       132
         GPE       0.60      0.71      0.65       299
JURISDICTION       0.45      0.61      0.52       153
         LAW       0.15      0.34      0.21        65
         ORG       0.35      0.30      0.32       221
      PERSON       0.51      0.68      0.58       186
   PROVISION       0.25      0.59      0.35        87
 TAX_CONCEPT       0.25      0.38      0.30       353
    TAX_TYPE       0.46      0.48      0.47        86

   micro avg       0.39      0.54      0.45      1688
   macro avg       0.40      0.54      0.45      1688
weighted avg       0.42      0.54      0.46      1688



In [70]:
results_classified("dictionary_fewshot_may.conll", gold_file =  r"C:\Users\M.Walavalkar\OneDrive - IBFD\Desktop\thesis-ner-manya-explore\datasets\my_data\final_dataset_1st_may.conll")

              precision    recall  f1-score   support

       COURT       0.37      0.72      0.49       106
        DATE       0.75      0.73      0.74       132
         GPE       0.67      0.75      0.71       299
JURISDICTION       0.57      0.71      0.63       153
         LAW       0.13      0.42      0.20        65
         ORG       0.42      0.45      0.43       221
      PERSON       0.46      0.53      0.49       186
   PROVISION       0.37      0.68      0.48        87
 TAX_CONCEPT       0.32      0.27      0.29       353
    TAX_TYPE       0.44      0.53      0.48        86

   micro avg       0.45      0.55      0.49      1688
   macro avg       0.45      0.58      0.50      1688
weighted avg       0.47      0.55      0.50      1688

